In [1]:
import json
import spacy 

In [2]:
def toSpacy(dataSet):
    spacy_data = []
    for entry in dataSet:
        text = str(entry['text']) 
        entities = [(start, end, label) for start, end, label in entry['label']]
        spacy_data.append((text, {"entities": entities}))
    return spacy_data

In [3]:
file_path = "dataset.jsonl"

DataSet =[]

# Leer todas las líneas como una lista y recorrerlas
with open(file_path, 'r') as file:
    lines = file.readlines()
    for line in lines:
        data = json.loads(line)
        DataSet.append(data)

spacyDataSet=toSpacy(DataSet)

In [4]:
#para arreglar las etiquetas que inician/terminan en espacios en blanco
i=0
for text, props in spacyDataSet:
    for index,row in enumerate(props["entities"]):
        start = row[0]
        end = row[1]
        while text[start] ==" ":
            start+=1
            i+=1
        
        while text[end-1] ==" ":
            end-=1
            i+=1
            
        my_list = list(row)

        # Modificar un elemento
        my_list[0] = start
        my_list[1] = end

        # Volver a convertir a tupla si es necesario
        props["entities"][index] = tuple(my_list)
print(i)




42


In [7]:
import spacy
from spacy.training import offsets_to_biluo_tags

def align_offsets_to_tokens(doc, entities):
    """
    Alinea los offsets de las entidades a los límites de los tokens generados por spaCy.
    """
    aligned_entities = []
    for start, end, label in entities:
        token_start = None
        token_end = None
        for token in doc:
            # Encontrar el token que contiene el inicio de la entidad
            if token.idx <= start < token.idx + len(token.text):
                token_start = token.idx
            # Encontrar el token que contiene el final de la entidad
            if token.idx < end <= token.idx + len(token.text):
                token_end = token.idx + len(token.text)
        # Si ambos límites están definidos, añadir la entidad ajustada
        if token_start is not None and token_end is not None:
            aligned_entities.append((token_start, token_end, label))
    return aligned_entities

In [8]:
nlp = spacy.blank("es")
 
        

for text, props in spacyDataSet:
    doc = nlp.make_doc(text)
    props["entities"]=align_offsets_to_tokens(doc, props["entities"])

In [10]:
import random
random.shuffle(spacyDataSet)


countTrainning = int(len(spacyDataSet) * 0.7)
countTest =  len(spacyDataSet)-countTrainning

training = spacyDataSet[:countTrainning]
test = spacyDataSet[:countTest]

print(len(spacyDataSet),countTrainning,countTest)

print(training[0])

10 7 3
('ses95leg372c999ant.mp4\nMuy buenas tardes.\nSaludar al personal del Congreso que nos permite entonces desarrollar estas peticiones de antecedentes a distintos organismos y administrativos del Estado.\nTiene la palabra el señor diputado Jorge Ratkep Chifferli, de acuerdo a lo que él quiera manifestar.\nPor favor, señor Ratkep, tiene la palabra.\nSolicito que se pueda oficiar al gobierno regional de la Araucanía, a la delegación regional de la región de la Araucanía, a la dirección de obras hidráulicas de la región de la Araucanía, como también al ministro de Obras Públicas, al subsecretario de Obras Públicas y también al director nacional de la dirección de obras hidráulicas por lo siguiente.\nHace ya bastante tiempo atrás se aprobó por parte del gobierno regional con asesoría de la dirección de obras hidráulicas un proyecto muy grande de agua potable rural en la comuna de Traiguén.\nEn ese momento era uno de los proyectos más grandes que se iba a realizar de agua potable rural

In [11]:
import spacy
from spacy.training.example import Example
from spacy.util import minibatch, compounding
import random

# Paso 2: Crear un modelo base o cargar uno existente
# Para entrenar desde cero:
#nlp = spacy.blank("es")  # Crear modelo vacío para español

nlp = spacy.load("es_core_news_lg")  #modelo pre entrenado


# Añadir componente NER al pipeline
if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner", last=True)
else:
    ner = nlp.get_pipe("ner")


# Paso 3: Añadir etiquetas personalizadas al modelo
for _, annotations in spacyDataSet:
    for ent in annotations.get("entities"):
        ner.add_label(ent[2])





# Paso 4: Entrenamiento del modelo
# Desactivar componentes no necesarias (por eficiencia)
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]
with nlp.disable_pipes(*other_pipes):  # Solo entrenar NER
    optimizer = nlp.begin_training()
    for i in range(100):  # Número de épocas (ajústalo según necesidad)
        random.shuffle(training)
        losses = {}
        # Crear lotes de entrenamiento
        batches = minibatch(training, size=compounding(4.0, 32.0, 1.001))
        for batch in batches:
            texts, annotations = zip(*batch)
            examples = [Example.from_dict(nlp.make_doc(text), ann) for text, ann in zip(texts, annotations)]
            nlp.update(examples, drop=0.4, losses=losses)
        print(f"Iteración {i+1}, Pérdidas: {losses}")

Iteración 1, Pérdidas: {'ner': 9556.864387512207}
Iteración 2, Pérdidas: {'ner': 9166.390571594238}
Iteración 3, Pérdidas: {'ner': 12131.26596069336}
Iteración 4, Pérdidas: {'ner': 11623.130912780762}
Iteración 5, Pérdidas: {'ner': 11141.422365188599}
Iteración 6, Pérdidas: {'ner': 10158.040546417236}
Iteración 7, Pérdidas: {'ner': 8246.701360702515}
Iteración 8, Pérdidas: {'ner': 5539.189972877502}
Iteración 9, Pérdidas: {'ner': 4237.5605363845825}
Iteración 10, Pérdidas: {'ner': 2945.0119671821594}
Iteración 11, Pérdidas: {'ner': 2982.8052349090576}
Iteración 12, Pérdidas: {'ner': 2007.0140824317932}
Iteración 13, Pérdidas: {'ner': 936.0535153746605}
Iteración 14, Pérdidas: {'ner': 917.9590281844139}
Iteración 15, Pérdidas: {'ner': 670.7373935692012}
Iteración 16, Pérdidas: {'ner': 624.3556093135849}
Iteración 17, Pérdidas: {'ner': 400.49250252521597}
Iteración 18, Pérdidas: {'ner': 647.6381230950356}
Iteración 19, Pérdidas: {'ner': 548.7997447997332}
Iteración 20, Pérdidas: {'ner': 

In [24]:
# Paso 5: Guardar el modelo entrenado
#nlp.to_disk("modelo_entrenado_blank")

nlp.to_disk("modelo_entrenado_es_core_news_lg")

In [ ]:
# 9. Probar el modelo entrenado
test_text = test[0][0]
doc = nlp(test_text)
for ent in doc.ents:
    print(ent.text, ent.label_)

el señor diputado Jorge Ratkep AUTOR
Solicito que se pueda oficiar EVENTO
al gobierno regional de la Araucanía DESTINO
a la delegación regional de la región de la Araucanía DESTINO
a la dirección de obras hidráulicas de la región de la Araucanía DESTINO
al ministro de Obras Públicas DESTINO
al subsecretario de Obras Públicas DESTINO
al director nacional de la dirección de obras hidráulicas DESTINO
solicito que los órganos EVENTO
el diputado Beltrán y AUTOR
el diputado don Bernardo Berger AUTOR
solicito que se oficie EVENTO
a la ministra de Obras Públicas DESTINO
a la seremía del Ministerio de Obras Públicas DESTINO
a la Dirección de Vialidad de la región de Los Ríos DESTINO
para abordar las siguientes acciones de manera prioritaria. MATERIA
para que no se ignore la falta de progreso en estas rutas, que son fundamentales para la conectividad, el desarrollo y la seguridad de la región. MATERIA
el diputado Juan Carlos Beltrán, AUTOR
al Presidente de la República DESTINO
solicito que se of

In [26]:
import spacy
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer

examples = []
for text, annotations in test:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annotations)
    examples.append(example)

# Evaluar el modelo
scorer = nlp.evaluate(examples)

# Mostrar métricas
print("Token-level metrics:")
print(f"Precision: {scorer['token_acc']:.3f}")
print(f"Recall: {scorer['ents_p']:.3f}")
print(f"F1: {scorer['ents_f']:.3f}")
print("Entity-level metrics:")
print(f"Entities Precision: {scorer['ents_p']:.3f}")
print(f"Entities Recall: {scorer['ents_r']:.3f}")
print(f"Entities F1: {scorer['ents_f']:.3f}")

Token-level metrics:
Precision: 1.000
Recall: 0.810
F1: 0.837
Entity-level metrics:
Entities Precision: 0.810
Entities Recall: 0.865
Entities F1: 0.837


# en Blanco
# nlp = spacy.blank("es") 


Token-level metrics:
Precision: 1.000
Recall: 0.681
F1: 0.671
Entity-level metrics:
Entities Precision: 0.681
Entities Recall: 0.662
Entities F1: 0.671


# pre entrenado
# nlp = spacy.load("es_core_news_lg")
Token-level metrics:
Precision: 1.000
Recall: 0.810
F1: 0.837
Entity-level metrics:
Entities Precision: 0.810
Entities Recall: 0.865
Entities F1: 0.837
